# Synthetic Claude QLoRA SFT

Training notebook pentru condiția 5: același model open-source mic, fine-tuned pe date sintetice RO chiropractor generate cu Claude.

Ce NU face notebook-ul:
- nu folosește `TEST_IDS`;
- nu rulează evaluarea finală pe test;
- nu modifică split-ul proiectului.

Output principal: adaptor LoRA salvat în `artifacts/ft/runs/<RUN_NAME>/adapter`.

In [2]:
# Dependencies. Runtime restart may be needed if Colab had older packages preloaded.
%pip -q install -U \
  "transformers>=4.57.0,<5" \
  "accelerate>=1.11.0" \
  "peft>=0.17.0" \
  "bitsandbytes>=0.45.0" \
  "liger-kernel>=0.5.0" \
  "datasets>=3.0.0" \
  "jsonschema" \
  "sentencepiece" \
  "protobuf"

In [35]:
# Repo checkout / refresh.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SoftNestSol/Medical-Notes.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Medical-Notes")
UPDATE_REPO = True

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    elif UPDATE_REPO:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    os.chdir(REPO_DIR)
else:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "src" / "data_split.py").exists():
            os.chdir(candidate)
            break

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
SOTA = SRC / "SOTA_EVALUATION"
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(SOTA))

# Avoid Colab extension timeouts when HF_TOKEN is not available from UI secrets.
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("WANDB_DISABLED", "true")
# Must be set before torch CUDA allocations; restart runtime after changing it.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print(f"IN_COLAB={IN_COLAB}")
print(f"ROOT={ROOT}")

IN_COLAB=True
ROOT=/content/Medical-Notes


In [ ]:
# Training config. Start with SMOKE_TEST=True if the runtime is uncertain.
from datetime import datetime

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
SYNTHETIC_ROOT = ROOT / "data" / "synthetic" / "Claude"
SEED = 20260623
RUN_NAME = f"qwen3-4b-synth-claude-{datetime.now().strftime('%Y%m%d-%H%M')}"

SMOKE_TEST = False
SMOKE_MAX_EXAMPLES = 32

# T4 15GB cannot reliably backprop Qwen3-4B at 3k context. 2304 is the
# next pragmatic step above 2048 when Liger fused CE is enabled.
MAX_SEQ_LENGTH = 2304
VAL_FRACTION = 0.08
NUM_TRAIN_EPOCHS = 2
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
USE_LIGER_KERNEL = True

RUN_DIR = ROOT / "artifacts" / "ft" / "runs" / RUN_NAME
DATASET_DIR = ROOT / "artifacts" / "ft" / "datasets"
DATA_JSONL = DATASET_DIR / "synthetic_claude_sft_messages.jsonl"
ADAPTER_DIR = RUN_DIR / "adapter"
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    NUM_TRAIN_EPOCHS = 1

print("RUN_NAME", RUN_NAME)
print("RUN_DIR", RUN_DIR)
print("SMOKE_TEST", SMOKE_TEST)

RUN_NAME qwen3-4b-synth-claude-20260622-2252
RUN_DIR /content/Medical-Notes/artifacts/ft/runs/qwen3-4b-synth-claude-20260622-2252
SMOKE_TEST True


In [ ]:
# GPU check.
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print("capability", f"{major}.{minor}")
    BF16_OK = major >= 8
    !nvidia-smi
else:
    BF16_OK = False
    raise RuntimeError("GPU runtime required for QLoRA training")

In [37]:
# Build SFT JSONL through repo script, so leakage/schema checks stay centralized.
from data_split import POOL_COMPLETE_PAIR_IDS, TEST_IDS, assert_no_test_leakage

assert_no_test_leakage(POOL_COMPLETE_PAIR_IDS, context="pool complete-pair sanity")
print("TEST_IDS", len(TEST_IDS), sorted(TEST_IDS))
print("POOL_COMPLETE_PAIR_IDS", len(POOL_COMPLETE_PAIR_IDS), sorted(POOL_COMPLETE_PAIR_IDS))

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "build_chiro_sft_jsonl.py"),
    "--synthetic-root", str(SYNTHETIC_ROOT),
    "--out", str(DATA_JSONL),
]
subprocess.run(cmd, check=True)
print(DATA_JSONL)
print(DATA_JSONL.read_text(encoding="utf-8").splitlines()[0][:1000])

TEST_IDS 18 ['audio15', 'audio16', 'audio17', 'audio20', 'audio22', 'audio25', 'audio27', 'audio28', 'audio30', 'audio31', 'audio32', 'audio34', 'audio35', 'audio36', 'audio37', 'audio5', 'audio6', 'audio7']
POOL_COMPLETE_PAIR_IDS 10 ['audio1', 'audio10', 'audio18', 'audio19', 'audio21', 'audio23', 'audio24', 'audio26', 'audio29', 'audio4']
/content/Medical-Notes/artifacts/ft/datasets/synthetic_claude_sft_messages.jsonl
{"messages": [{"role": "system", "content": "Ești un sistem de extragere de informații clinice din transcrierile consultațiilor unui cabinet de chiropractică din România.\n\nPrimești o transcriere (dialog terapeut-pacient) și returnezi UN SINGUR obiect JSON cu informațiile extrase.\n\nREGULA FUNDAMENTALĂ: dacă o informație NU este rostită explicit în conversație, câmpul rămâne gol (null sau listă goală). Nu deduce. Nu presupune. Nu completa pe baza intuiției clinice.\n\nSchema exactă:\n{\n  \"motivul_prezentarii\": <string sau null>,\n  \"evaluarea_durerii_vas\": <int 0

In [38]:
# Load + split synthetic data. This validation split is only for training sanity, not final evaluation.
from datasets import load_dataset

if not DATA_JSONL.exists():
    print(f"DATA_JSONL missing, rebuilding: {DATA_JSONL}")
    cmd = [
        sys.executable,
        str(ROOT / "scripts" / "build_chiro_sft_jsonl.py"),
        "--synthetic-root", str(SYNTHETIC_ROOT),
        "--out", str(DATA_JSONL),
    ]
    subprocess.run(cmd, check=True)

raw = load_dataset("json", data_files=str(DATA_JSONL), split="train")
raw = raw.shuffle(seed=SEED)
if SMOKE_TEST:
    raw = raw.select(range(min(SMOKE_MAX_EXAMPLES, len(raw))))

splits = raw.train_test_split(test_size=VAL_FRACTION, seed=SEED)
train_raw = splits["train"]
val_raw = splits["test"]

print("raw", len(raw))
print("train", len(train_raw))
print("val", len(val_raw))
if not SMOKE_TEST and len(raw) < 300:
    raise RuntimeError(
        f"Expected full Claude synthetic dataset, got only {len(raw)} rows. "
        "Check SMOKE_TEST, DATA_JSONL, and repo/data checkout."
    )
print(train_raw[0]["messages"][-1]["content"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

raw 32
train 29
val 3
{"motivul_prezentarii":"Durere umăr drept apărută după efort fizic (mutat mobilă), de aproximativ două săptămâni. Durere la mișcare, mai ales la ridicarea brațului, cu episoade nocturne.","evaluarea_durerii_vas":5,"localizarea_durerii":["umar_dr"],"localizarea_durerii_alta":null,"antecedente":[],"antecedente_altele":null,"medicatie_actuala":[{"denumire":"ibuprofen","doza":null}],"evaluare_functionala_initiala":"Diagnostic:\n- tendinită coif rotatori umăr drept\nObservații funcționale:\n- mobilit


In [ ]:
# Model + tokenizer + LoRA.
# Liger is important on T4: standard HF loss materializes full-sequence logits
# and often OOMs on Qwen3's large vocabulary.
if USE_LIGER_KERNEL:
    import inspect
    import importlib.util
    if importlib.util.find_spec("liger_kernel") is None:
        print("liger_kernel missing; installing now. If import still fails, restart runtime and rerun from top.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "liger-kernel>=0.5.0"], check=True)
    try:
        from liger_kernel.transformers import apply_liger_kernel_to_qwen3
    except Exception as exc:
        raise RuntimeError("USE_LIGER_KERNEL=True but liger-kernel import failed. Restart runtime after installing liger-kernel, then rerun from the first cell.") from exc

    params = inspect.signature(apply_liger_kernel_to_qwen3).parameters
    liger_kwargs = {}
    # Pick fused CE when available; Liger asserts if cross_entropy and
    # fused_linear_cross_entropy are both True.
    if "fused_linear_cross_entropy" in params:
        liger_kwargs["fused_linear_cross_entropy"] = True
    elif "cross_entropy" in params:
        liger_kwargs["cross_entropy"] = True
    for key in ["rms_norm", "rope", "swiglu"]:
        if key in params:
            liger_kwargs[key] = True
    apply_liger_kernel_to_qwen3(**liger_kwargs)
    print("Liger kernel enabled", liger_kwargs)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

HF_TOKEN_ARG = False  # Qwen is public. For gated models, login manually and set this to True.
compute_dtype = torch.bfloat16 if BF16_OK else torch.float16

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN_ARG,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    token=HF_TOKEN_ARG,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Tokenization with prompt masking: loss is computed only on assistant JSON.
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}

def render_prompt(messages):
    return tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )

def render_full(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        **CHAT_TEMPLATE_KWARGS,
    )

def tokenize_example(example):
    messages = example["messages"]
    prompt_text = render_prompt(messages)
    full_text = render_full(messages)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    # Most chat templates make prompt_ids an exact prefix. If not, use a conservative fallback.
    if full_ids[: len(prompt_ids)] != prompt_ids:
        assistant_text = messages[-1]["content"] + (tokenizer.eos_token or "")
        full_ids = tokenizer(prompt_text + assistant_text, add_special_tokens=False)["input_ids"]
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "length": len(full_ids),
        "supervised_tokens": sum(1 for token in labels if token != -100),
    }

train_tok = train_raw.map(tokenize_example, remove_columns=train_raw.column_names)
val_tok = val_raw.map(tokenize_example, remove_columns=val_raw.column_names)

before_train = len(train_tok)
before_val = len(val_tok)
train_tok = train_tok.filter(lambda row: row["length"] <= MAX_SEQ_LENGTH and row["supervised_tokens"] > 0)
val_tok = val_tok.filter(lambda row: row["length"] <= MAX_SEQ_LENGTH and row["supervised_tokens"] > 0)

print("train length range before filter", min(train_tok["length"]), max(train_tok["length"]))
print("val length range before filter", min(val_tok["length"]), max(val_tok["length"]))
print("train kept", len(train_tok), "/", before_train)
print("val kept", len(val_tok), "/", before_val)
if len(train_tok) == 0 or len(val_tok) == 0:
    raise RuntimeError(
        f"No examples left after token length filtering: train={len(train_tok)}/{before_train}, "
        f"val={len(val_tok)}/{before_val}, MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. "
        "Raise MAX_SEQ_LENGTH or use a less tiny SMOKE_MAX_EXAMPLES."
    )
if not SMOKE_TEST and len(train_tok) < 100:
    raise RuntimeError(
        f"Full run would train on only {len(train_tok)} examples after filtering. "
        "This is too small for condition 5; raise MAX_SEQ_LENGTH on a larger GPU or shorten examples."
    )
print("max train length after filter", max(train_tok["length"]))
print("max val length after filter", max(val_tok["length"]))

In [ ]:
# Padding collator for causal LM labels.
from dataclasses import dataclass
from typing import Any

@dataclass
class CausalLMCollator:
    tokenizer: Any

    def __call__(self, features):
        max_len = max(len(feature["input_ids"]) for feature in features)
        pad_id = self.tokenizer.pad_token_id
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for feature in features:
            pad_len = max_len - len(feature["input_ids"])
            batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(feature["attention_mask"] + [0] * pad_len)
            batch["labels"].append(feature["labels"] + [-100] * pad_len)
        return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

collator = CausalLMCollator(tokenizer)

In [ ]:
# Train.
import inspect
from transformers import Trainer, TrainingArguments

steps_per_epoch = max(1, len(train_tok) // (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
eval_steps = max(10, steps_per_epoch // 2)
logging_steps = max(5, eval_steps // 2)
print("effective batch size", PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("estimated optimizer steps/epoch", steps_per_epoch)
print("estimated total optimizer steps", steps_per_epoch * NUM_TRAIN_EPOCHS)
if not SMOKE_TEST and steps_per_epoch * NUM_TRAIN_EPOCHS < 20:
    raise RuntimeError(
        "Full run has suspiciously few optimizer steps. "
        "Expected tens of steps for ~344 synthetic examples."
    )

args_kwargs = dict(
    output_dir=str(RUN_DIR / "checkpoints"),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=logging_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
    bf16=BF16_OK,
    fp16=not BF16_OK,
)

# Transformers renamed evaluation_strategy -> eval_strategy in newer versions.
ta_params = inspect.signature(TrainingArguments.__init__).parameters
strategy_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"
args_kwargs[strategy_key] = "steps"
args_kwargs["save_strategy"] = "steps"

training_args = TrainingArguments(**args_kwargs)
import gc
gc.collect()
torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

train_result = trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.state.save_to_json(str(RUN_DIR / "trainer_state.json"))
print(train_result)
print("adapter saved", ADAPTER_DIR)

In [ ]:
# Sanity generation on synthetic validation examples only.
import json
import re
from parser import ParseError, SchemaError, parse_note

model.eval()

def generate_raw(messages, max_new_tokens=768):
    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def canonical_vas(value):
    if isinstance(value, list):
        return sorted(value)
    return value

def field_match(pred, gold, field):
    if pred is None:
        return False
    if field in {"localizarea_durerii", "antecedente"}:
        return sorted(pred.get(field, [])) == sorted(gold.get(field, []))
    if field == "evaluarea_durerii_vas":
        return canonical_vas(pred.get(field)) == canonical_vas(gold.get(field))
    if field == "medicatie_actuala":
        return pred.get(field) == gold.get(field)
    return pred.get(field) == gold.get(field)

FIELDS = [
    "motivul_prezentarii",
    "evaluarea_durerii_vas",
    "localizarea_durerii",
    "localizarea_durerii_alta",
    "antecedente",
    "antecedente_altele",
    "medicatie_actuala",
    "evaluare_functionala_initiala",
]

sanity_n = min(8, len(val_raw))
parse_ok = 0
field_totals = {field: 0 for field in FIELDS}
rows = []
for i in range(sanity_n):
    messages = val_raw[i]["messages"]
    gold = json.loads(messages[-1]["content"])
    raw_output = generate_raw(messages)
    try:
        parsed = parse_note(raw_output)
        status = "ok"
        parse_ok += 1
        matches = {field: field_match(parsed, gold, field) for field in FIELDS}
        for field, ok in matches.items():
            field_totals[field] += int(ok)
    except (ParseError, SchemaError) as exc:
        parsed = None
        status = f"fail: {type(exc).__name__}: {exc}"
        matches = {field: False for field in FIELDS}
    rows.append({"idx": i, "status": status, "raw": raw_output, "parsed": parsed, "gold": gold, "field_matches": matches})
    print("=" * 80)
    print("idx", i, status)
    print("field_matches", matches)
    print(raw_output[:1200])

(RUN_DIR / "sanity_generations.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"parse/schema sanity: {parse_ok}/{sanity_n}")
print("field exact sanity:", {field: f"{count}/{sanity_n}" for field, count in field_totals.items()})
print("wrote", RUN_DIR / "sanity_generations.json")

In [ ]:
# Export manifest + adapter zip.
import json
import shutil
from datetime import datetime, timezone

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"Adapter directory not found: {ADAPTER_DIR}")

manifest = {
    "run_name": RUN_NAME,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "synthetic_root": str(SYNTHETIC_ROOT),
    "data_jsonl": str(DATA_JSONL),
    "run_dir": str(RUN_DIR),
    "adapter_dir": str(ADAPTER_DIR),
    "sanity_generations": str(RUN_DIR / "sanity_generations.json"),
    "smoke_test": SMOKE_TEST,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "use_liger_kernel": USE_LIGER_KERNEL,
}

manifest_path = RUN_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
adapter_zip = shutil.make_archive(str(ADAPTER_DIR), "zip", ADAPTER_DIR)

print("manifest", manifest_path)
print("adapter zip", adapter_zip)
print("run dir", RUN_DIR)

In [ ]:
# Backup run artifacts to Google Drive.
# Copies adapter, adapter zip, trainer_state, sanity_generations, and run_manifest.
try:
    from google.colab import drive  # type: ignore
    IN_COLAB_FOR_BACKUP = True
except Exception:
    IN_COLAB_FOR_BACKUP = False

if not IN_COLAB_FOR_BACKUP:
    print("Not running in Colab; skip Drive backup.")
else:
    drive_mount = Path("/content/drive")
    mydrive = drive_mount / "MyDrive"
    if not mydrive.exists():
        try:
            drive.mount(str(drive_mount), force_remount=False)
        except Exception as exc:
            print("Drive mount failed:", repr(exc))
            print("Fallback: use the direct download cell, or run this notebook from the Colab web UI and retry Drive mount.")

    if mydrive.exists():
        drive_root = mydrive / "Medical-Notes" / "ft-runs"
        drive_target = drive_root / RUN_NAME
        drive_target.parent.mkdir(parents=True, exist_ok=True)
        if drive_target.exists():
            shutil.rmtree(drive_target)
        shutil.copytree(RUN_DIR, drive_target)
        print("Drive backup complete:", drive_target)
        print("Files:")
        for path in sorted(drive_target.rglob("*")):
            if path.is_file():
                print(path.relative_to(drive_target), path.stat().st_size)
    else:
        print("Drive is not mounted; backup skipped.")

In [ ]:
# Optional direct downloads from Colab browser.
# Set DOWNLOAD_ARTIFACTS=True to download through the browser.
# This is the recommended fallback when Drive auth fails in VS Code/Colab extension.
DOWNLOAD_ARTIFACTS = True

if DOWNLOAD_ARTIFACTS:
    from google.colab import files  # type: ignore
    files.download(str(ADAPTER_DIR) + ".zip")
    files.download(str(RUN_DIR / "run_manifest.json"))
    sanity_path = RUN_DIR / "sanity_generations.json"
    if sanity_path.exists():
        files.download(str(sanity_path))
else:
    print("DOWNLOAD_ARTIFACTS=False; artifacts are available in RUN_DIR and Drive backup cell.")